# Iron March — 00: Extracción y Exploración
Caso 3 del curso de análisis forense de leaks underground.
Iron March fue un foro neonazi activo de 2011 a 2017.
Dataset: ~18K usuarios, ~194K posts, ~55K mensajes privados, 138 foros.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import networkx as nx
from collections import Counter

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_forum, RESULTS_DIR, DATA_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

IM_RESULTS = RESULTS_DIR / 'ironmarch'
IM_RESULTS.mkdir(parents=True, exist_ok=True)
IM_ZIP = DATA_DIR / 'Far Right Forum' / 'IronMarch_2019.11.zip'
print(f'Datos:      {DATA_DIR}')
print(f'Resultados: {IM_RESULTS}')


## 1. Estructura del dataset

Iron March usa IPS 4.x (Invision Power Suite). El dump SQL contiene 5 tablas principales.

### Esquema relacional
```
forum → thread → post → user
                          ↑↑
                   private_message
```

**Cardinalidades aproximadas**: ~138 foros, ~14K threads, ~194K posts, ~18K usuarios, ~55K PMs.
Nótese que no hay subforos — la jerarquía es plana (todos los foros son de primer nivel).

In [ ]:
# Cargar el dump — el parser detecta automáticamente el formato IPS
raw = load_forum(IM_ZIP)

# Extraer tablas principales con nombres normalizados
forums   = raw.get('forum', pd.DataFrame())
threads  = raw.get('thread', pd.DataFrame())
posts    = raw.get('post', pd.DataFrame())
users    = raw.get('user', pd.DataFrame())
pms      = raw.get('private_message', pd.DataFrame())

# userid debe ser numérico — filas con texto en userid son errores del parser
for df in [posts, threads, pms]:
    for col in ['userid', 'sender_id', 'receiver_id']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.dropna(subset=[col], inplace=True)
            df[col] = df[col].astype(int).astype(str)

for col in ['userid']:
    if col in users.columns:
        users[col] = pd.to_numeric(users[col], errors='coerce')
        users.dropna(subset=[col], inplace=True)
        users[col] = users[col].astype(int).astype(str)

# Asegurar datetimes con timezone UTC
for df, col in [(posts, 'dateline'), (users, 'joindate'), (threads, 'dateline')]:
    if col in df.columns and len(df) > 0:
        df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

# Resumen de tablas
summary = {
    'Tabla': ['forum', 'thread', 'post', 'user', 'private_message'],
    'Filas': [len(forums), len(threads), len(posts), len(users), len(pms)],
    'Columnas clave': [
        str(list(forums.columns[:5])),
        str(list(threads.columns[:5])),
        str(list(posts.columns[:5])),
        str(list(users.columns[:5])),
        str(list(pms.columns[:5])),
    ],
}
print(pd.DataFrame(summary).to_string(index=False))

print(f'\nForos/secciones:   {len(forums):,}')
print(f'Hilos:             {len(threads):,}')
print(f'Posts:             {len(posts):,}')
print(f'Usuarios:          {len(users):,}')
print(f'Mensajes privados: {len(pms):,}')

if 'dateline' in posts.columns:
    valid = posts['dateline'].dropna()
    if len(valid):
        print(f'\nRango temporal: {valid.min().strftime("%Y-%m-%d")} → {valid.max().strftime("%Y-%m-%d")}')


## 2. Análisis Exploratorio (EDA)

Primera pregunta: ¿qué tenemos realmente aquí? Antes de cualquier modelo, hay que entender el dato.

In [ ]:
# Distribución de posts por usuario (power-law)
# En casi todos los foros: pocos usuarios escriben muchísimo, la mayoría casi nada
if 'userid' in posts.columns:
    post_counts = posts.groupby('userid').size().sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(post_counts.values, bins=60, color='#E94E4E', alpha=0.85)
    axes[0].set_title('Distribución de posts por usuario')
    axes[0].set_xlabel('Posts')
    axes[0].set_ylabel('Usuarios')

    # Log-log revela la power-law
    bins = np.logspace(0, np.log10(post_counts.max() + 1), 40)
    axes[1].hist(post_counts.values, bins=bins, color='#E94E4E', alpha=0.85)
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_title('Power-law (escala log-log)')
    axes[1].set_xlabel('Posts (log)')
    axes[1].set_ylabel('Usuarios (log)')

    plt.suptitle('IronMarch — actividad de usuarios', y=1.01)
    plt.tight_layout()
    plt.show()

    # Estadísticas clave
    print(f'Usuarios con 1 post:      {(post_counts == 1).sum():,} ({(post_counts == 1).mean()*100:.1f}%)')
    print(f'Usuarios con ≥10 posts:   {(post_counts >= 10).sum():,} ({(post_counts >= 10).mean()*100:.1f}%)')
    print(f'Usuarios con ≥100 posts:  {(post_counts >= 100).sum():,}')
    print(f'Top 10% genera {post_counts.nlargest(int(len(post_counts)*0.1)).sum() / post_counts.sum()*100:.0f}% de los posts')

    # Top 20 usuarios más activos
    uid_to_name = dict(zip(users['userid'], users['username'])) if 'username' in users.columns else {}
    top_users = (
        posts.groupby('userid').size()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    top_users.columns = ['userid', 'posts']
    top_users['username'] = top_users['userid'].map(uid_to_name)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(
        top_users['username'].fillna(top_users['userid'].astype(str))[::-1],
        top_users['posts'][::-1],
        color='#E94E4E', alpha=0.85,
    )
    ax.set_xlabel('Posts')
    ax.set_title('Top 20 usuarios más activos — IronMarch')
    plt.tight_layout()
    plt.show()


In [ ]:
# Evolución temporal — posts mensuales con eventos clave
EVENTS = {
    '2011-09': 'Fundación del foro',
    '2017-11': 'Cierre (Operation Payback)',
}

if 'dateline' in posts.columns:
    valid_posts = posts.dropna(subset=['dateline'])
    valid_posts = valid_posts[
        (valid_posts['dateline'].dt.year >= 2011) &
        (valid_posts['dateline'].dt.year <= 2018)
    ]
    monthly = valid_posts.groupby(valid_posts['dateline'].dt.to_period('M')).size()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(range(len(monthly)), monthly.values, color='#E94E4E', linewidth=1.5)
    ax.fill_between(range(len(monthly)), monthly.values, alpha=0.2, color='#E94E4E')

    month_index = {str(p): i for i, p in enumerate(monthly.index)}
    for event_ym, label in EVENTS.items():
        if event_ym in month_index:
            idx = month_index[event_ym]
            ax.axvline(idx, color='#FFD700', alpha=0.7, linestyle='--', linewidth=1.5)
            ax.text(idx + 0.3, monthly.max() * 0.9, label,
                    rotation=90, fontsize=8, color='#FFD700', va='top')

    tick_step = max(1, len(monthly) // 14)
    ax.set_xticks(range(0, len(monthly), tick_step))
    ax.set_xticklabels(
        [str(monthly.index[i]) for i in range(0, len(monthly), tick_step)],
        rotation=45, fontsize=8,
    )
    ax.set_title('IronMarch — posts mensuales')
    ax.set_ylabel('Posts / mes')
    plt.tight_layout()
    plt.show()


In [ ]:
# Heatmap: actividad por hora UTC y día de semana
# Revela la franja horaria dominante de los usuarios
if 'dateline' in posts.columns:
    df_tz = posts.dropna(subset=['dateline']).copy()
    df_tz['hour']    = df_tz['dateline'].dt.hour
    df_tz['weekday'] = df_tz['dateline'].dt.weekday

    heatmap_data = df_tz.groupby(['weekday', 'hour']).size().unstack(fill_value=0)
    day_names = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
    heatmap_data.index = [day_names[i] for i in heatmap_data.index]

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, linewidths=0.1,
                cbar_kws={'label': 'Posts'})
    ax.set_title('IronMarch — actividad por hora UTC y día de la semana')
    ax.set_xlabel('Hora UTC')
    plt.tight_layout()
    plt.show()


In [ ]:
# Top 20 foros/secciones más activos
# posts solo tiene threadid — hay que joinear con threads para obtener forumid
if len(threads) > 0 and 'forumid' in threads.columns and 'threadid' in posts.columns:
    thread_to_forum = threads.set_index('threadid')['forumid'].to_dict()
    posts_with_forum = posts.copy()
    posts_with_forum['forumid'] = posts_with_forum['threadid'].map(thread_to_forum)

    if len(forums) > 0:
        fname_col = next((c for c in forums.columns if c == 'name'), None)
        fid_col   = next((c for c in forums.columns if c == 'forumid'), None)
        if fname_col and fid_col:
            fid_to_name = dict(zip(forums[fid_col].astype(str), forums[fname_col]))
            posts_with_forum['forumid'] = posts_with_forum['forumid'].astype(str)

            forum_counts = (
                posts_with_forum.groupby('forumid').size()
                .sort_values(ascending=False)
                .head(20)
                .reset_index()
            )
            forum_counts.columns = ['forumid', 'posts']
            forum_counts['name'] = forum_counts['forumid'].map(fid_to_name).fillna('(desconocido)')

            fig, ax = plt.subplots(figsize=(10, 8))
            ax.barh(forum_counts['name'][::-1], forum_counts['posts'][::-1],
                    color='#E94E4E', alpha=0.85)
            ax.set_xlabel('Posts')
            ax.set_title('Top 20 foros más activos — IronMarch')
            plt.tight_layout()
            plt.show()
        else:
            print(f'Columnas de forums: {list(forums.columns)}')
    else:
        print('Sin tabla de foros cargada.')
else:
    print(f'Columnas de posts: {list(posts.columns)}')
    print(f'Columnas de threads: {list(threads.columns) if len(threads) > 0 else "vacío"}')


## 3. Red de co-participación pública

Grafo donde los nodos son usuarios y las aristas conectan a quienes participaron en el mismo hilo.
El **betweenness centrality** identifica los brokers — usuarios que conectan subcomunidades distintas dentro del foro.

Filtramos hilos con más de 100 posts para que los mega-threads no dominen la topología del grafo.

In [ ]:
# Construir grafo de co-participación en hilos
# Excluimos mega-threads (> 100 posts) para no distorsionar la red
G_public = nx.Graph()

thread_col = 'threadid' if 'threadid' in posts.columns else None

if thread_col and 'userid' in posts.columns:
    df_graph = posts[['threadid', 'userid']].copy()
    df_graph['threadid'] = df_graph['threadid'].astype(str)
    df_graph['userid']   = df_graph['userid'].astype(str)
    df_graph = df_graph[df_graph['threadid'] != 'nan']

    # Filtrar mega-threads
    thread_sizes = df_graph.groupby('threadid').size()
    valid_threads = thread_sizes[thread_sizes <= 100].index
    df_graph = df_graph[df_graph['threadid'].isin(valid_threads)]
    print(f'Hilos válidos (≤100 posts): {len(valid_threads):,} de {len(thread_sizes):,}')

    thread_users = df_graph.groupby('threadid')['userid'].apply(list)

    edge_weights = Counter()
    for participants in thread_users:
        unique = list(set(participants))
        for i in range(len(unique)):
            for j in range(i + 1, min(i + 10, len(unique))):
                pair = tuple(sorted([unique[i], unique[j]]))
                edge_weights[pair] += 1

    for (u, v), w in edge_weights.items():
        G_public.add_edge(u, v, weight=w)

    print(f'Grafo público: {G_public.number_of_nodes():,} nodos, {G_public.number_of_edges():,} aristas')

    # Calcular betweenness centrality (con sampling para grafos grandes)
    uid_to_name = dict(zip(users['userid'], users['username'])) if 'username' in users.columns else {}

    degree_cent = nx.degree_centrality(G_public)
    k_sample = min(300, G_public.number_of_nodes())
    print(f'Calculando betweenness (k={k_sample})...')
    btw_cent = nx.betweenness_centrality(G_public, k=k_sample, normalized=True, weight='weight')

    top_brokers = sorted(btw_cent.items(), key=lambda x: x[1], reverse=True)[:10]
    print('\nTop 10 brokers (betweenness centrality):')
    for uid, score in top_brokers:
        name = uid_to_name.get(uid, str(uid))
        deg  = degree_cent.get(uid, 0)
        print(f'  {name:<25} btw={score:.4f}  deg={deg:.4f}')
else:
    print(f'Columnas disponibles en posts: {list(posts.columns)}')


In [ ]:
# Visualización interactiva con Plotly — zoom, hover, pannable
# Solo top 150 nodos por betweenness para evitar solapamiento
import plotly.graph_objects as go

if G_public.number_of_nodes() > 0:
    largest_cc = max(nx.connected_components(G_public), key=len)
    G_main = G_public.subgraph(largest_cc).copy()

    # Seleccionar top 150 por betweenness dentro del componente principal
    top_nodes = sorted(
        [n for n in G_main.nodes() if n in btw_cent],
        key=lambda n: btw_cent[n], reverse=True
    )[:150]
    G_viz = G_main.subgraph(top_nodes).copy()

    pos = nx.spring_layout(G_viz, k=1.2, iterations=80, seed=42)

    # Aristas
    edge_x, edge_y = [], []
    for u, v in G_viz.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.4, color='rgba(150,150,150,0.3)'),
        hoverinfo='none',
    )

    # Nodos
    node_x     = [pos[n][0] for n in G_viz.nodes()]
    node_y     = [pos[n][1] for n in G_viz.nodes()]
    node_btw   = [btw_cent.get(n, 0) for n in G_viz.nodes()]
    node_deg   = [G_viz.degree(n) for n in G_viz.nodes()]
    node_names = [uid_to_name.get(n, f'uid:{n}') for n in G_viz.nodes()]
    node_sizes = [b * 60 + 6 for b in node_btw]

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(
            size=node_sizes,
            color=node_btw,
            colorscale='YlOrRd',
            colorbar=dict(title='Betweenness'),
            line=dict(width=0.5, color='white'),
        ),
        text=[name if btw_cent.get(n, 0) > sorted(node_btw, reverse=True)[min(12, len(node_btw)-1)] else ''
              for n, name in zip(G_viz.nodes(), node_names)],
        textposition='top center',
        textfont=dict(size=9, color='white'),
        hovertext=[f'{name}<br>btw={btw_cent.get(n,0):.4f}<br>conexiones={G_viz.degree(n)}'
                   for n, name in zip(G_viz.nodes(), node_names)],
        hoverinfo='text',
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title='Red IronMarch — top 150 nodos (hover para detalles, zoom libre)',
            template='plotly_dark',
            showlegend=False,
            width=1000, height=750,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=50, b=20),
        )
    )
    fig.show()
else:
    print('Grafo vacío.')


## 4. Red de mensajes privados

~55K mensajes privados entre miembros — dato infrecuente en leaks. La red de PMs revela quién habla con quién **fuera del espacio público** del foro.

Pregunta clave: ¿los brokers públicos son los mismos que los privados?

In [ ]:
# Construir grafo dirigido de mensajes privados
G_pm = nx.DiGraph()

sender_col   = next((c for c in pms.columns if 'from' in c.lower() or 'author' in c.lower() or 'sender' in c.lower()), None)
receiver_col = next((c for c in pms.columns if 'to' in c.lower() or 'recipient' in c.lower() or 'receiver' in c.lower()), None)

print(f'Columna sender:   {sender_col}')
print(f'Columna receiver: {receiver_col}')

if sender_col and receiver_col:
    pm_edges = pms[[sender_col, receiver_col]].dropna()
    pm_edges = pm_edges[pm_edges[sender_col] != pm_edges[receiver_col]]

    for _, row in pm_edges.iterrows():
        s, r = str(row[sender_col]), str(row[receiver_col])
        if G_pm.has_edge(s, r):
            G_pm[s][r]['weight'] += 1
        else:
            G_pm.add_edge(s, r, weight=1)

    print(f'\nGrafo PMs: {G_pm.number_of_nodes():,} nodos, {G_pm.number_of_edges():,} aristas')

    # Betweenness sobre versión no-dirigida
    G_pm_und = G_pm.to_undirected()
    k_pm = min(200, G_pm_und.number_of_nodes())
    btw_pm = nx.betweenness_centrality(G_pm_und, k=k_pm, normalized=True)

    top_pm_brokers = sorted(btw_pm.items(), key=lambda x: x[1], reverse=True)[:10]
    print('\nTop 10 brokers en red de PMs:')
    for uid, score in top_pm_brokers:
        name = uid_to_name.get(uid, str(uid))
        print(f'  {name:<25} btw={score:.4f}')

    # Intersección con brokers públicos
    top_public_uids = {uid for uid, _ in top_brokers[:10]}
    top_pm_uids     = {uid for uid, _ in top_pm_brokers[:10]}
    overlap = top_public_uids & top_pm_uids
    print(f'\nBrokers en ambas redes: {len(overlap)}')
    for uid in overlap:
        print(f'  {uid_to_name.get(uid, str(uid))}')
else:
    print(f'Columnas disponibles en PMs: {list(pms.columns)}')


In [ ]:
# Visualización interactiva de la red de PMs con Plotly
# Color = in-degree (quién recibe más mensajes)
import plotly.graph_objects as go

if G_pm.number_of_nodes() > 0:
    # in-degree normalizado para colorear
    in_deg = dict(G_pm.in_degree())

    top_pm_nodes = sorted(G_pm_und.degree(), key=lambda x: x[1], reverse=True)[:150]
    top_pm_set   = {n for n, _ in top_pm_nodes}
    G_pm_viz     = G_pm_und.subgraph(top_pm_set).copy()

    pos = nx.spring_layout(G_pm_viz, k=0.8, iterations=60, seed=42)

    # Aristas
    edge_x, edge_y = [], []
    for u, v in G_pm_viz.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.5, color='rgba(150,150,150,0.25)'),
        hoverinfo='none',
    )

    # Nodos
    node_x      = [pos[n][0] for n in G_pm_viz.nodes()]
    node_y      = [pos[n][1] for n in G_pm_viz.nodes()]
    node_indeg  = [in_deg.get(n, 0) for n in G_pm_viz.nodes()]
    node_btw_pm = [btw_pm.get(n, 0) for n in G_pm_viz.nodes()]
    node_names  = [uid_to_name.get(n, f'uid:{n}') for n in G_pm_viz.nodes()]
    node_sizes  = [b * 80 + 6 for b in node_btw_pm]

    top8_pm = {uid for uid, _ in top_pm_brokers[:8]}

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(
            size=node_sizes,
            color=node_indeg,
            colorscale='Plasma',
            colorbar=dict(title='In-degree (PMs recibidos)'),
            line=dict(width=0.5, color='white'),
        ),
        text=[name if n in top8_pm else ''
              for n, name in zip(G_pm_viz.nodes(), node_names)],
        textposition='top center',
        textfont=dict(size=9, color='white'),
        hovertext=[
            f'{name}<br>in-degree={in_deg.get(n,0)}<br>btw={btw_pm.get(n,0):.4f}<br>conexiones PM={G_pm_viz.degree(n)}'
            for n, name in zip(G_pm_viz.nodes(), node_names)
        ],
        hoverinfo='text',
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title='Red de mensajes privados — IronMarch (top 150, hover para detalles)',
            template='plotly_dark',
            showlegend=False,
            width=1000, height=750,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=50, b=20),
        )
    )
    fig.show()
else:
    print('Sin datos de PM para construir la red.')


## 5. Análisis temporal de actividad

Calculamos el volumen semanal de posts y marcamos los períodos con actividad anómalamente alta (z-score > 2σ). Esto nos permite correlacionar picos de actividad con eventos externos.

In [ ]:
# Picos de actividad semanal con z-score
from scipy import stats as scipy_stats

if 'dateline' in posts.columns:
    valid_posts = posts.dropna(subset=['dateline']).copy()
    valid_posts = valid_posts[
        (valid_posts['dateline'].dt.year >= 2011) &
        (valid_posts['dateline'].dt.year <= 2018)
    ]

    weekly = valid_posts.groupby(
        valid_posts['dateline'].dt.to_period('W')
    ).size()

    # Z-score
    z_scores = scipy_stats.zscore(weekly.values)
    spike_mask = z_scores > 2.0

    fig, ax = plt.subplots(figsize=(16, 5))
    x = range(len(weekly))
    ax.plot(x, weekly.values, color='#E94E4E', linewidth=1, alpha=0.9)

    # Resaltar picos
    for i, (is_spike, count) in enumerate(zip(spike_mask, weekly.values)):
        if is_spike:
            ax.axvspan(i - 0.5, i + 0.5, color='#FFD700', alpha=0.25)

    ax.set_title('IronMarch — actividad semanal con picos (z > 2σ en amarillo)')
    ax.set_ylabel('Posts / semana')

    tick_step = max(1, len(weekly) // 14)
    ax.set_xticks(range(0, len(weekly), tick_step))
    ax.set_xticklabels(
        [str(weekly.index[i]) for i in range(0, len(weekly), tick_step)],
        rotation=45, fontsize=7,
    )
    plt.tight_layout()
    plt.show()

    # Imprimir top períodos de pico
    spike_periods = [(str(weekly.index[i]), weekly.values[i], z_scores[i])
                     for i in range(len(weekly)) if spike_mask[i]]
    spike_periods.sort(key=lambda x: x[2], reverse=True)
    print(f'Semanas con z > 2σ: {len(spike_periods)}')
    print('\nTop 10 picos de actividad:')
    for period, count, z in spike_periods[:10]:
        print(f'  {period}  posts={count:,}  z={z:.2f}')
else:
    print('Sin columna dateline en posts.')


## 6. Detección de idioma

Antes de aplicar cualquier modelo — Burrows' Delta, NER, topic modeling — hay que verificar en qué idioma están los posts. Estos modelos son idioma-específicos: las function words de Delta son distintas en inglés y alemán, los prompts de NER asumen inglés, BERTopic aprende vocabulario del corpus.

> **Hipótesis**: IronMarch era un foro en inglés con audiencia global, pero ¿qué porcentaje real de posts es inglés?

In [ ]:
from langdetect import detect, LangDetectException

def detect_lang(text):
    try:
        return detect(str(text)[:500])
    except LangDetectException:
        return 'unknown'

# Muestra estratificada — 2000 posts es suficiente para distribución estable
sample_size = min(2000, len(posts))
posts_sample = posts.dropna(subset=['pagetext']).sample(sample_size, random_state=42).copy()

print(f"Detectando idioma en {len(posts_sample):,} posts...")
posts_sample['lang'] = posts_sample['pagetext'].apply(detect_lang)

lang_dist = posts_sample['lang'].value_counts()
lang_pct  = (lang_dist / lang_dist.sum() * 100).round(1)

print("\nDistribución de idioma (muestra):")
for lang, pct in lang_pct.head(10).items():
    bar = '█' * int(pct / 2)
    print(f"  {lang:<8} {pct:>5.1f}%  {bar}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lang_pct.head(8).index, lang_pct.head(8).values, color='#E94E4E', alpha=0.85)
ax.set_ylabel('% posts')
ax.set_title('IronMarch — distribución de idioma (muestra n=2000)')
ax.axhline(y=80, color='#FFD700', linestyle='--', alpha=0.5, label='80%')
plt.tight_layout()
plt.show()

dominant     = lang_pct.index[0]
dominant_pct = lang_pct.iloc[0]
print(f"\nIdioma dominante: {dominant} ({dominant_pct:.1f}%)")
if dominant == 'en' and dominant_pct >= 80:
    print("✓ Pipeline en inglés aplicable: Burrows' Delta, NER y BERTopic sin ajustes.")
else:
    print(f"⚠️  Idioma dominante no es inglés — revisar pipeline antes de continuar con notebooks 02 y 03.")